In [18]:
import shutil
import tempfile
from pathlib import Path

from pystac import Catalog, ItemCollection, get_stac_version

In [9]:
cat = Catalog.from_file("stac-examples/catalog.json")

In [10]:
cat

<Catalog id=sen4x-examples>

In [11]:
list(cat.get_items())

[<Item id=S2AGRI_S4C_TILL_PRD_S2_20261008T071443_V20190101T000000_20191231T000000>,
 <Item id=S2AGRI_S4C_L4C_PRD_S2_20261008T081633_V20190101T000000_20191231T235959>,
 <Item id=S2AGRI_S4C_L4B_PRD_S2_20261007T070259_V20190301T000000_20191231T000000>,
 <Item id=S2AGRI_S4C_L4A_PRD_S2_20261007T060248_V20190101T000000_20191231T000000>,
 <Item id=S2AGRI_S4C_HETEROGENEITY_PRD_S2_20261008T105820_V20190101T000000_20191231T235959>,
 <Item id=S2AGRI_S4C_BARESOIL_PRD_S2_20261008T071917_V20190101T000000_20191231T000000>]

In [12]:
item = cat.get_item("S2AGRI_S4C_L4B_PRD_S2_20261007T070259_V20190301T000000_20191231T000000")

In [13]:
item

<Item id=S2AGRI_S4C_L4B_PRD_S2_20261007T070259_V20190301T000000_20191231T000000>

In [14]:
item.assets

{'mowing': <Asset href=../../../../../../../../../../mnt/archive/test_general/s4c_l4b/S2AGRI_S4C_L4B_PRD_S2_20261007T070259_V20190301T000000_20191231T000000/VECTOR_DATA/SEN4CAP_L4B_S1_S2_MowingDetection.shp>,
 'metadata': <Asset href=../../../../../../../../../../mnt/archive/test_general/s4c_l4b/S2AGRI_S4C_L4B_PRD_S2_20261007T070259_V20190301T000000_20191231T000000/S2AGRI_S4C_L4B_MTD_V20190301T000000_20191231T000000.xml>}

In [16]:
item.assets["mowing"]

<Asset href=../../../../../../../../../../mnt/archive/test_general/s4c_l4b/S2AGRI_S4C_L4B_PRD_S2_20261007T070259_V20190301T000000_20191231T000000/VECTOR_DATA/SEN4CAP_L4B_S1_S2_MowingDetection.shp>

In [17]:
cat.get_all_items()

In [19]:
ItemCollection(cat.get_all_items())

In [21]:
list(cat.get_items())[0].assets["tillage"].href

'../../../../../../../../../../mnt/archive/test_general/s4c_l4c_tillage/S2AGRI_S4C_TILL_PRD_S2_20261008T071443_V20190101T000000_20191231T000000/TILES/tillage_detection.csv'

In [29]:
from html import escape
from urllib.parse import unquote, urlsplit

from IPython.display import HTML, display
from pystac import Item, ItemCollection

try:
    import ipywidgets as widgets
    from ipyleaflet import GeoJSON, Map, Rectangle
except ImportError:
    widgets = None
    GeoJSON = Map = Rectangle = None


def show_assets(items: Item | ItemCollection) -> None:
    """Render grouped STAC items and assets, with optional ipyleaflet previews."""
    if isinstance(items, Item):
        item_list = [items]
        expression_for = lambda i, key: f"items.assets[{key!r}]"
    elif isinstance(items, ItemCollection):
        item_list = list(items.items)
        expression_for = lambda i, key: f"items.items[{i}].assets[{key!r}]"
    else:
        raise TypeError("items must be a pystac.Item or pystac.ItemCollection")

    def geometry_bounds(geometry):
        """Get [west, south, east, north] from GeoJSON coordinates."""
        positions = []

        def visit(value):
            if isinstance(value, (list, tuple)):
                if (
                    len(value) >= 2
                    and isinstance(value[0], (int, float))
                    and isinstance(value[1], (int, float))
                ):
                    positions.append((value[0], value[1]))
                else:
                    for child in value:
                        visit(child)

        if geometry.get("type") == "GeometryCollection":
            for child in geometry.get("geometries", []):
                bounds = geometry_bounds(child)
                if bounds:
                    positions.extend(((bounds[0], bounds[1]), (bounds[2], bounds[3])))
        else:
            visit(geometry.get("coordinates", []))

        if not positions:
            return None

        longitudes, latitudes = zip(*positions)
        return (
            min(longitudes), min(latitudes),
            max(longitudes), max(latitudes),
        )

    def make_map(item):
        if Map is None:
            return None

        bbox = item.bbox
        if bbox and len(bbox) == 4:
            west, south, east, north = bbox
        elif bbox and len(bbox) == 6:
            west, south, east, north = bbox[0], bbox[1], bbox[3], bbox[4]
        elif item.geometry:
            bounds = geometry_bounds(item.geometry)
            if not bounds:
                return None
            west, south, east, north = bounds
        else:
            return None

        map_widget = Map(
            center=[(south + north) / 2, (west + east) / 2],
            zoom=2,
            scroll_wheel_zoom=False,
            layout=widgets.Layout(width="360px", height="240px"),
        )
        map_bounds = ((south, west), (north, east))

        if item.geometry:
            layer = GeoJSON(
                data={
                    "type": "Feature",
                    "geometry": item.geometry,
                    "properties": {},
                },
                style={"color": "#d62728", "fillOpacity": 0.15},
            )
        else:
            layer = Rectangle(
                bounds=map_bounds,
                color="#d62728",
                fill_opacity=0.15,
            )

        map_widget.add_layer(layer)
        map_widget.fit_bounds(map_bounds)
        return map_widget

    for item_index, item in enumerate(item_list):
        item_id = escape(str(item.id))
        collection = escape(str(item.collection_id or ""))
        datetime = escape(str(item.properties.get("datetime") or ""))

        asset_rows = []
        for key, asset in item.assets.items():
            href = str(asset.href)
            path = urlsplit(href.replace("\\", "/")).path.rstrip("/")
            filename = unquote(path.rsplit("/", 1)[-1]) or href
            scheme = urlsplit(href).scheme.lower()

            if scheme in ("", "http", "https", "file"):
                file_html = (
                    f"<a href='{escape(href, quote=True)}' "
                    "style='overflow-wrap:anywhere'>"
                    f"{escape(filename)}</a>"
                )
            else:
                file_html = escape(filename)

            expression = expression_for(item_index, key)
            title = escape(str(asset.title or ""))

            asset_rows.append(
                "<tr>"
                f"<td>{escape(str(key))}</td>"
                f"<td>{file_html}</td>"
                f"<td>{title}</td>"
                "<td>"
                f"<button data-copy='{escape(expression, quote=True)}' "
                "onclick=\"(async b => {"
                "try { await navigator.clipboard.writeText(b.dataset.copy); "
                "b.title = 'Copied'; } "
                "catch (_) { window.prompt('Copy expression:', b.dataset.copy); }"
                "})(this)\">Copy</button>"
                "</td>"
                "</tr>"
            )

        if asset_rows:
            assets_html = (
                "<div style='margin:0.5em 0 0.5em 1.5em'>"
                "<table style='border-collapse:collapse;width:100%'>"
                "<thead><tr><th>Asset</th><th>File</th><th>Title</th><th></th>"
                "</tr></thead>"
                f"<tbody>{''.join(asset_rows)}</tbody>"
                "</table></div>"
            )
        else:
            assets_html = (
                "<div style='margin-left:1.5em'><em>No assets</em></div>"
            )

        item_html = (
            "<section style='margin:1em 0;padding:0.75em;"
            "border:1px solid #888;border-radius:0.4em'>"
            f"<div><strong>{item_id}</strong></div>"
            f"<div style='opacity:0.75'>{collection} · {datetime}</div>"
            f"{assets_html}</section>"
        )

        map_widget = make_map(item)
        if map_widget is not None:
            display(
                widgets.VBox(
                    [widgets.HTML(value=item_html), map_widget],
                    layout=widgets.Layout(width="100%"),
                )
            )
        else:
            display(HTML(item_html))

In [30]:
from pystac import ItemCollection

items = ItemCollection(cat.get_all_items())
show_assets(items)

Asset,File,Title,
tillage,tillage_detection.csv,Tillage detection results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
metadata,S2AGRI_S4C_TILL_MTD_V20190101T000000_20191231T000000.xml,Product manifest,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"


Asset,File,Title,
na,Sen4CAP_L4C_NA_CZE_2019_CSV.csv,Unspecified practice results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
nfc,Sen4CAP_L4C_NFC_CZE_2019_CSV.csv,Nitrogen-fixing crops results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
fallow,Sen4CAP_L4C_Fallow_CZE_2019_CSV.csv,Fallow land results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
na_plot,Sen4CAP_L4C_NA_CZE_2019_PLOT.xml,Unspecified practice plot data,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
metadata,S2AGRI_S4C_L4C_MTD_V20190101T000000_20191231T235959.xml,Product manifest,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
nfc_plot,Sen4CAP_L4C_NFC_CZE_2019_PLOT.xml,Nitrogen-fixing crops plot data,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
catch_crop,Sen4CAP_L4C_CatchCrop_CZE_2019_CSV.csv,Catch crop practice results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
fallow_plot,Sen4CAP_L4C_Fallow_CZE_2019_PLOT.xml,Fallow land plot data,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
catch_crop_plot,Sen4CAP_L4C_CatchCrop_CZE_2019_PLOT.xml,Catch crop plot data,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"


Asset,File,Title,
mowing,SEN4CAP_L4B_S1_S2_MowingDetection.shp,Grassland mowing detection,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
metadata,S2AGRI_S4C_L4B_MTD_V20190301T000000_20191231T000000.xml,Product manifest,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"


Asset,File,Title,
lut,CropType_LUT.csv,Crop lookup table,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
metadata,S2AGRI_S4C_L4A_MTD_V20190101T000000_20191231T000000.xml,Product manifest,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
validation,Data_validation_final_1007_0601.ipc,Validation dataset,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
calibration,Data_calibration_final_after_smote_1007_0602.ipc,Training features after SMOTE,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
crop_summary,Crop_types_summary_1007_0601.csv,Crop type statistics,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
accuracy_plot,Accuracy_metrics_plot_1007_0602.png,Accuracy metrics plot,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
classification,CropType.gpkg,Classification output,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
predictions_all,Parcels_all_with_predictions_1007_0602.csv,All parcel predictions,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
accuracy_metrics,Accuracy_metrics_1007_0602.csv,Accuracy metrics,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
confusion_matrix,Confusion_matrix_1007_0602.csv,Classification confusion matrix,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"


Asset,File,Title,
metadata,S2AGRI_S4C_HETEROGENEITY_MTD_V20190101T000000_20191231T235959.xml,Product manifest,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m1,L4D_HeteS1_2_2019_1.csv,S1 heterogeneity month 1,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m2,L4D_HeteS1_2_2019_2.csv,S1 heterogeneity month 2,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m3,L4D_HeteS1_2_2019_3.csv,S1 heterogeneity month 3,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m4,L4D_HeteS1_2_2019_4.csv,S1 heterogeneity month 4,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m5,L4D_HeteS1_2_2019_5.csv,S1 heterogeneity month 5,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m6,L4D_HeteS1_2_2019_6.csv,S1 heterogeneity month 6,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m7,L4D_HeteS1_2_2019_7.csv,S1 heterogeneity month 7,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m8,L4D_HeteS1_2_2019_8.csv,S1 heterogeneity month 8,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
hete_s1_m9,L4D_HeteS1_2_2019_9.csv,S1 heterogeneity month 9,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"


Asset,File,Title,
metadata,S2AGRI_S4C_BARESOIL_MTD_V20190101T000000_20191231T000000.xml,Product manifest,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
markers_s1,L4E_BS_MarkersS1.csv,S1 bare soil markers,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
markers_s2,L4E_BS_MarkersS2.csv,S2 bare soil markers,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
results_s1,L4E_BS_S1results.csv,S1 bare soil detection results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
results_s2,L4E_BS_S2results.csv,S2 bare soil detection results,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
markers_all,L4E_BS_MarkersAll.csv,Combined bare soil markers,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
importance_s1,BSmodelS1featuresimportance.png,S1 feature importance plot,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
importance_s2,BSmodelS2featuresimportance.png,S2 feature importance plot,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
calibration_s1,L4E_BS_CalibrationS1.csv,S1 training features,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
calibration_s2,L4E_BS_CalibrationS2.csv,S2 training features,"{try { await navigator.clipboard.writeText(b.dataset.copy); b.title = 'Copied'; } catch (_) { window.prompt('Copy expression:', b.dataset.copy); }})(this)"">Copy"
